# Workbook 5: Optimise with any prebuilt oracle

This notebook is the **practical BO workbook**. The aim is to let you run Bayesian optimisation on any prebuilt oracle with only a small amount of editing, while still leaving room for deeper customization if desired.

**Recommended first use:**

1. Leave the defaults alone and run the notebook top-to-bottom once.
2. Change only `ORACLE_KEY` and `OPTIMIZE_TARGET`.
3. Then change one BO choice at a time, rather than many at once.

Workbook 4 explains the BO choices in more detail. Workbook 5 is where you apply them.


In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scripts import get_oracle_meta, list_oracles, load_oracle
import scripts.bo_optimisation as bo_optimisation

bo_optimisation = importlib.reload(bo_optimisation)

best_so_far = bo_optimisation.best_so_far
build_named_setup_plot_frame = bo_optimisation.build_named_setup_plot_frame
build_search_space = bo_optimisation.build_search_space
make_default_acquisition_scorer = bo_optimisation.make_default_acquisition_scorer
merge_comparison_overrides = bo_optimisation.merge_comparison_overrides
run_bo = bo_optimisation.run_bo
run_named_setup_comparisons = bo_optimisation.run_named_setup_comparisons
summarize_named_setup_comparisons = bo_optimisation.summarize_named_setup_comparisons
validate_comparison_setups = bo_optimisation.validate_comparison_setups

registry_rows = []
for key in list_oracles():
    meta = get_oracle_meta(key)
    registry_rows.append(
        {
            "oracle_key": key,
            "label": meta["label"],
            "default_target": meta["default_target"],
            "targets": ", ".join(meta["targets"]),
        }
    )

display(pd.DataFrame(registry_rows))


## Main configuration

Edit this cell first. Everything in the rest of the notebook uses these settings.

The first block defines the main BO setup. The final block, `COMPARISON_SETUPS`, lets you define your own named BO variants as **overrides of the main setup**.

Then, to optimise your BO: create multiple entries in `COMPARISON_SETUPS` where certain properties are changed. By defining them here, you can keep track of your progress. Use `SINGLE_RUN_SETUP` to choose whether the single-run diagnostic section uses the main setup or one of these named variants.

`BATCH_SIZE` experiments are selected together using Kriging believer (notebook 4), before their results are returned. Keep `N_CANDIDATES >= BATCH_SIZE`.

Maximum budget: `N_INITIAL + N_ITER * BATCH_SIZE`. The default comparison gives both setups 42 experiments: 12 initial, then either 30 batches of 1 or 10 batches of 3. Keep budgets equal when editing settings.


In [ ]:
ORACLE_KEY = "dataset_a1"  # Options: "dataset_a1", "dataset_b1", "dataset_b2", "dataset_b3", "dataset_b4", "dataset_c1", "dataset_d1", "dataset_e1", "dataset_e2".
OPTIMIZE_TARGET = None  # Options: None, or one of the targets listed for your ORACLE_KEY in the table above.
# Note: the current BO setup treats OPTIMIZE_TARGET as a maximise objective.

SEARCH_SPACE_OVERRIDE = {
    # Restrict the allowed search space here if needed.
    # Example categorical subset:
    # "ligand": ["L0", "L2", "L4"],
    # Example continuous bounds:
    # "temperature": (50.0, 100.0),
}

ENCODING_MODE = "onehot"  # Options: "onehot", "ordinal", "custom".
CUSTOM_CATEGORICAL_MAPS = {
    # Only used if ENCODING_MODE = "custom".
    # Example:
    # "ligand": {
    #     "L0": [0.10, 0.40],
    #     "L1": [0.20, 0.35],
    # }
}

INIT_STRATEGY = "random"  # Options: "random", "latin_hypercube", "space_filling", "grid".
KERNEL_NAME = "matern52"  # GP kernel: matern52, matern32, rbf, rq, periodic.
ACQ_FUNCTION = "ei"  # Acquisition rule: ei, pi, ucb, greedy, thompson.
EI_XI = 0.01  # For EI/PI: larger values ask for clearer improvement, so BO explores more.
UCB_BETA = 2.0  # For UCB: larger values put more weight on uncertainty, so BO explores more.

N_INITIAL = 12  # Number of initial experiments before BO starts.
N_ITER = 30  # Number of BO batches after the initial set.
BATCH_SIZE = 1  # Experiments per batch; 1 gives sequential BO.
N_CANDIDATES = 2000  # Candidate points sampled and scored at each BO step.
SINGLE_RUN_SEED = 42  # Random seed for the single BO campaign.
N_RUNS = 5  # Number of repeated BO campaigns in the comparison summary.
COMPARE_AFTER_EXPERIMENT = 10  # Extra comparison metric: compare best-so-far after this many experiments.
SHOW_COMPARISON_STD_BANDS = True  # Options: True, False.
BASE_SEED = 42  # Starting seed for repeated BO; later runs increment from this.
SAVE_EXPERIMENT_LOGS = False  # Options: True, False.
LOG_DIR = "bo_experiment_logs"  # Folder for saved logs.

COMPARISON_SETUPS = {
    # Define your own named BO variants here as overrides of the main setup.
    # Keep the dataset, target, and search space fixed; compare BO choices only.
    # Allowed override keys: ENCODING_MODE, CUSTOM_CATEGORICAL_MAPS,
    # INIT_STRATEGY, KERNEL_NAME, ACQ_FUNCTION, EI_XI, UCB_BETA, N_INITIAL, N_ITER,
    # BATCH_SIZE, N_CANDIDATES.
    # "ucb_more_explore": {"ACQ_FUNCTION": "ucb", "UCB_BETA": 4.0},
    # "latin_hypercube_init": {"INIT_STRATEGY": "latin_hypercube"},
    "batch_of_three": {"BATCH_SIZE": 3, "N_ITER": 10},  # Same 42-experiment budget as main.
    # "larger_initial_set": {"N_INITIAL": 16, "N_ITER": 26},
    # "smaller_initial_set": {"N_INITIAL": 8, "N_ITER": 34},
}
SINGLE_RUN_SETUP = "main"  # Options here: "main", "batch_of_three" etc.

In [ ]:
ORACLE_META = get_oracle_meta(ORACLE_KEY)
bundle = load_oracle(ORACLE_KEY)
spec = bundle.spec

if OPTIMIZE_TARGET is None:
    OPTIMIZE_TARGET = ORACLE_META["default_target"]
if OPTIMIZE_TARGET not in spec.targets:
    raise ValueError(f"Choose OPTIMIZE_TARGET from {spec.targets}")
if ENCODING_MODE not in {"onehot", "ordinal", "custom"}:
    raise ValueError("ENCODING_MODE must be 'onehot', 'ordinal', or 'custom'")
if ENCODING_MODE == "custom" and not CUSTOM_CATEGORICAL_MAPS:
    raise ValueError("Provide CUSTOM_CATEGORICAL_MAPS when ENCODING_MODE = 'custom'")
if INIT_STRATEGY not in {"random", "latin_hypercube", "space_filling", "grid"}:
    raise ValueError("Unknown INIT_STRATEGY")
if KERNEL_NAME not in {"matern52", "matern32", "rbf", "rq", "periodic"}:
    raise ValueError("Unknown KERNEL_NAME")
if ACQ_FUNCTION not in {"ei", "pi", "ucb", "greedy", "thompson"}:
    raise ValueError("Unknown ACQ_FUNCTION")
if N_INITIAL < 1:
    raise ValueError("N_INITIAL must be at least 1")
if N_ITER < 0:
    raise ValueError("N_ITER must be at least 0")
if N_CANDIDATES < 1:
    raise ValueError("N_CANDIDATES must be at least 1")
if N_RUNS < 1:
    raise ValueError("N_RUNS must be at least 1")
if COMPARE_AFTER_EXPERIMENT < 1:
    raise ValueError("COMPARE_AFTER_EXPERIMENT must be at least 1")
if not isinstance(SHOW_COMPARISON_STD_BANDS, bool):
    raise ValueError("SHOW_COMPARISON_STD_BANDS must be True or False")

validated_comparison_setups = validate_comparison_setups(COMPARISON_SETUPS)
SEARCH_SPACE = build_search_space(spec, SEARCH_SPACE_OVERRIDE)

BASE_WORKBOOK_CONFIG = {
    "SEARCH_SPACE_OVERRIDE": SEARCH_SPACE_OVERRIDE,
    "ENCODING_MODE": ENCODING_MODE,
    "CUSTOM_CATEGORICAL_MAPS": CUSTOM_CATEGORICAL_MAPS if ENCODING_MODE == "custom" else None,
    "INIT_STRATEGY": INIT_STRATEGY,
    "KERNEL_NAME": KERNEL_NAME,
    "ACQ_FUNCTION": ACQ_FUNCTION,
    "EI_XI": EI_XI,
    "UCB_BETA": UCB_BETA,
    "N_INITIAL": N_INITIAL,
    "N_ITER": N_ITER,
    "BATCH_SIZE": BATCH_SIZE,
    "N_CANDIDATES": N_CANDIDATES,
    "N_RUNS": N_RUNS,
    "BASE_SEED": BASE_SEED,
    "SAVE_EXPERIMENT_LOGS": SAVE_EXPERIMENT_LOGS,
    "LOG_DIR": LOG_DIR,
}

available_single_run_setups = ["main", *validated_comparison_setups.keys()]
if SINGLE_RUN_SETUP not in available_single_run_setups:
    raise ValueError(f"SINGLE_RUN_SETUP must be one of {available_single_run_setups}")

ACTIVE_SINGLE_RUN_OVERRIDES = {} if SINGLE_RUN_SETUP == "main" else validated_comparison_setups[SINGLE_RUN_SETUP]
ACTIVE_SINGLE_RUN_CONFIG = merge_comparison_overrides(BASE_WORKBOOK_CONFIG, ACTIVE_SINGLE_RUN_OVERRIDES)
ACTIVE_SINGLE_RUN_SEARCH_SPACE = build_search_space(
    spec,
    ACTIVE_SINGLE_RUN_CONFIG["SEARCH_SPACE_OVERRIDE"],
)
ACTIVE_SCORE_CANDIDATES = make_default_acquisition_scorer(
    str(ACTIVE_SINGLE_RUN_CONFIG["ACQ_FUNCTION"]),
    ei_xi=float(ACTIVE_SINGLE_RUN_CONFIG["EI_XI"]),
    ucb_beta=float(ACTIVE_SINGLE_RUN_CONFIG["UCB_BETA"]),
)
ACTIVE_SCORE_LABEL = f"built-in:{ACTIVE_SINGLE_RUN_CONFIG['ACQ_FUNCTION']}"

settings_summary = pd.DataFrame(
    [
        {"setting": "dataset_label", "value": ORACLE_META["label"]},
        {"setting": "valid_targets", "value": ", ".join(spec.targets)},
        {"setting": "chosen_target", "value": OPTIMIZE_TARGET},
        {"setting": "objective_direction", "value": "maximize"},
        {"setting": "main_total_budget", "value": N_INITIAL + N_ITER * BATCH_SIZE},
        {"setting": "main_batch_size", "value": BATCH_SIZE},
        {"setting": "single_run_batch_size", "value": ACTIVE_SINGLE_RUN_CONFIG["BATCH_SIZE"]},
        {"setting": "main_encoding_mode", "value": ENCODING_MODE},
        {"setting": "main_kernel_name", "value": KERNEL_NAME},
        {"setting": "main_acq_function", "value": ACQ_FUNCTION},
        {"setting": "single_run_setup", "value": SINGLE_RUN_SETUP},
        {"setting": "single_run_total_budget", "value": int(ACTIVE_SINGLE_RUN_CONFIG["N_INITIAL"]) + int(ACTIVE_SINGLE_RUN_CONFIG["N_ITER"]) * ACTIVE_SINGLE_RUN_CONFIG["BATCH_SIZE"]},
        {"setting": "single_run_encoding_mode", "value": ACTIVE_SINGLE_RUN_CONFIG["ENCODING_MODE"]},
        {"setting": "single_run_kernel_name", "value": ACTIVE_SINGLE_RUN_CONFIG["KERNEL_NAME"]},
        {"setting": "single_run_acq_function", "value": ACTIVE_SINGLE_RUN_CONFIG["ACQ_FUNCTION"]},
        {"setting": "active_scorer", "value": ACTIVE_SCORE_LABEL},
        {"setting": "comparison_setups", "value": len(validated_comparison_setups)},
        {"setting": "compare_after_experiment", "value": COMPARE_AFTER_EXPERIMENT},
        {"setting": "show_comparison_std_bands", "value": SHOW_COMPARISON_STD_BANDS},
    ]
)

search_space_summary = pd.DataFrame(
    [
        {
            "feature": feature,
            "type": "categorical" if feature in spec.categorical else "continuous",
            "search_space": ACTIVE_SINGLE_RUN_SEARCH_SPACE[feature],
        }
        for feature in spec.features
    ]
)

display(settings_summary)
display(search_space_summary)
print(f"Single-run diagnostics will use setup: {SINGLE_RUN_SETUP}")
print("Pick one dataset and stay with it for a session so that changes in BO behavior are easier to interpret.")

if validated_comparison_setups:
    comparison_preview = pd.DataFrame(
        [
            {
                "setup_name": setup_name,
                "changes_from_main": ", ".join(f"{key}={value!r}" for key, value in overrides.items()),
            }
            for setup_name, overrides in validated_comparison_setups.items()
        ]
    )
    display(comparison_preview)
else:
    print("No named comparison setups defined yet. Add them in COMPARISON_SETUPS when you want to compare BO choices.")


## Run one BO setup

This is the simplest path through the notebook: run one BO campaign with the current settings, inspect the experiment log, then look at the trajectory plots. By default it uses `SINGLE_RUN_SETUP = "main"`, but you can point it at any named setup from `COMPARISON_SETUPS` if you want to inspect that variant in more detail.

When reading the log:

- rows with `source = initial_set` belong to the starting design, before BO starts using the GP
- rows with `source = bo` were chosen by the acquisition rule


In [ ]:
single_run = run_bo(
    bundle,
    target=OPTIMIZE_TARGET,
    search_space_override=ACTIVE_SINGLE_RUN_CONFIG["SEARCH_SPACE_OVERRIDE"],
    categorical_mode=ACTIVE_SINGLE_RUN_CONFIG["ENCODING_MODE"],
    custom_categorical_maps=ACTIVE_SINGLE_RUN_CONFIG["CUSTOM_CATEGORICAL_MAPS"],
    initial_strategy=ACTIVE_SINGLE_RUN_CONFIG["INIT_STRATEGY"],
    kernel_name=ACTIVE_SINGLE_RUN_CONFIG["KERNEL_NAME"],
    score_candidates=ACTIVE_SCORE_CANDIDATES,
    n_initial=int(ACTIVE_SINGLE_RUN_CONFIG["N_INITIAL"]),
    n_iter=int(ACTIVE_SINGLE_RUN_CONFIG["N_ITER"]),
    batch_size=ACTIVE_SINGLE_RUN_CONFIG["BATCH_SIZE"],
    n_candidates=int(ACTIVE_SINGLE_RUN_CONFIG["N_CANDIDATES"]),
    random_state=SINGLE_RUN_SEED,
    save_logs=bool(ACTIVE_SINGLE_RUN_CONFIG["SAVE_EXPERIMENT_LOGS"]),
    log_dir=str(ACTIVE_SINGLE_RUN_CONFIG["LOG_DIR"]),
    log_name=f"{ORACLE_KEY}_{SINGLE_RUN_SETUP}_single_run.csv",
)

print(f"Single-run setup: {SINGLE_RUN_SETUP}")
print(
    f"Encoding={ACTIVE_SINGLE_RUN_CONFIG['ENCODING_MODE']}, "
    f"kernel={ACTIVE_SINGLE_RUN_CONFIG['KERNEL_NAME']}, "
    f"acquisition={ACTIVE_SINGLE_RUN_CONFIG['ACQ_FUNCTION']}"
)
print(f"Best {OPTIMIZE_TARGET}: {single_run.best_value:.3f}")
print("Best conditions:")
print(single_run.best_conditions)
display(single_run.log)


In [ ]:
best_curve = best_so_far(single_run.log)
observed_values = single_run.log["oracle_value"].to_numpy(dtype=float)
simple_regret = float(np.max(best_curve)) - best_curve
x_axis = np.arange(1, len(best_curve) + 1)

fig, axes = plt.subplots(3, 1, figsize=(8, 10), sharex=True)
axes[0].plot(x_axis, best_curve, marker="o")
axes[0].set_ylabel(f"Best {OPTIMIZE_TARGET}")
axes[0].set_title(f"Single-run BO: {ORACLE_META['label']} ({SINGLE_RUN_SETUP})")

axes[1].plot(x_axis, observed_values, color="0.75", linewidth=1.5)
for source_name, group in single_run.log.groupby("source", sort=False):
    axes[1].scatter(
        group["experiment"],
        group["oracle_value"],
        label=source_name,
        s=45,
    )
axes[1].set_ylabel(f"Observed {OPTIMIZE_TARGET}")
axes[1].set_title("Observed target at each experiment")
axes[1].legend(title="source")

axes[2].plot(x_axis, simple_regret, marker="o", color="C1")
axes[2].set_xlabel("Experiment index")
axes[2].set_ylabel("Simple regret")
axes[2].set_title("Single-run regret")

plt.tight_layout()
plt.show()


**How to read these plots**

- **Best-so-far** shows the best objective value found up to each experiment.
- **Observed target at each experiment** shows the actual value returned at each step, so you can see whether improvement came from steady gains or a few jumps.
- **Simple regret** shows how far the current best is from the best value found in that run.
- One run is useful for intuition, but is not enough to judge whether a BO setup is robust.


## Compare named BO setups

Use `COMPARISON_SETUPS` when you want to compare BO choices without changing the dataset or target. Each named setup is an override of the main config.

This section always includes `main` as the baseline. The comparison table reports both final performance and **best-so-far by experiment `COMPARE_AFTER_EXPERIMENT`** so you can compare not only where setups finish, but also how quickly they get there. The comparison plots use mean curves across repeated runs, and `SHOW_COMPARISON_STD_BANDS` controls whether +/- 1 std shading is shown around those means. In the regret plot here, regret is measured relative to the best final value found anywhere among the compared setups; these two plots are the most commonly seen plots used to compare the progress of BO, and effectively show the same information.

Logs identify each `batch` and its `batch_position`; batch 0 is the initial design. Experiment-index curves order results retrospectively within each batch. These results become available together, so intermediate points are not model updates. Compare experiments used separately from the number of rounds needed.


In [ ]:
comparison_results = run_named_setup_comparisons(
    bundle,
    target=OPTIMIZE_TARGET,
    base_config=BASE_WORKBOOK_CONFIG,
    comparison_setups=validated_comparison_setups,
    base_setup_name="main",
)
comparison_summary = summarize_named_setup_comparisons(
    comparison_results,
    compare_after_experiment=COMPARE_AFTER_EXPERIMENT,
)
comparison_plot_frame = build_named_setup_plot_frame(comparison_results)

if "COMPARISON_RUN_LIBRARY" not in globals():
    COMPARISON_RUN_LIBRARY = {}
COMPARISON_RUN_LIBRARY.update(comparison_results)

display(comparison_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for setup_name, group in comparison_plot_frame.groupby("setup_name", sort=False):
    line_best, = axes[0].plot(group["experiment_index"], group["mean_best"], label=setup_name)
    line_regret, = axes[1].plot(group["experiment_index"], group["mean_regret"], label=setup_name)

    if SHOW_COMPARISON_STD_BANDS:
        axes[0].fill_between(
            group["experiment_index"],
            group["mean_best"] - group["std_best"],
            group["mean_best"] + group["std_best"],
            color=line_best.get_color(),
            alpha=0.12,
        )
        axes[1].fill_between(
            group["experiment_index"],
            group["mean_regret"] - group["std_regret"],
            group["mean_regret"] + group["std_regret"],
            color=line_regret.get_color(),
            alpha=0.12,
        )

axes[0].set_xlabel("Experiment index")
axes[0].set_ylabel(f"Mean best {OPTIMIZE_TARGET}")
axes[0].set_title("Comparison: mean best-so-far")
axes[0].legend()

axes[1].set_xlabel("Experiment index")
axes[1].set_ylabel("Mean regret vs comparison best")
axes[1].set_title("Comparison: mean regret")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Comparison results currently stored in memory:")
print(list(COMPARISON_RUN_LIBRARY.keys()))


**How to use the comparison section**

Good comparisons usually change **one idea at a time**:

- acquisition rule
- GP kernel
- initial design strategy
- batch size, while holding the total experiment budget fixed
- amount of exploration
- initial-vs-BO budget split

If several setups look similar, that is often useful information: the BO workflow may be fairly robust to those choices.

It is even possible to use BO to optimise BO itself, though its not expected that you do that here.


## Optional analysis: Runs before x% yield

This section is considered optional.

It's also possible to define your own **analysis metric** for judging BO campaigns, as comparing multiple runs can be tricky, especially if they look similar.

This example asks: **how many experiments did each setup need to reach a chosen target value?** The single-run result is shown separately from the repeated-run comparison summary so that the output stays easy to read.

There are numerous possible metrics you could use and develop.


In [ ]:
TARGET_THRESHOLD = 80  # Example: 80. Leave as None if you do not want threshold-based analysis.

def first_experiment_to_reach(log_df, threshold, value_col="oracle_value"):
    hits = np.flatnonzero(log_df[value_col].to_numpy(dtype=float) >= float(threshold))
    return int(hits[0] + 1) if len(hits) else pd.NA

def display_threshold_value(value):
    numeric = float(value)
    return int(numeric) if numeric.is_integer() else numeric

if TARGET_THRESHOLD is None:
    print("Set TARGET_THRESHOLD to a numeric value if you want to compare campaigns by a chemistry goal.")
else:
    shown_threshold = display_threshold_value(TARGET_THRESHOLD)

    single_hit = first_experiment_to_reach(single_run.log, TARGET_THRESHOLD)
    single_threshold_summary = pd.DataFrame(
        [
            {
                "setup_name": SINGLE_RUN_SETUP,
                "threshold": shown_threshold,
                "first_experiment_to_reach": single_hit,
            }
        ]
    )
    single_threshold_summary["first_experiment_to_reach"] = single_threshold_summary["first_experiment_to_reach"].astype("Int64")

    comparison_rows = []
    for setup_name, payload in comparison_results.items():
        hit_steps = [
            first_experiment_to_reach(result.log, TARGET_THRESHOLD)
            for result in payload["run_results"]
        ]
        finite_hits = [int(step) for step in hit_steps if pd.notna(step)]
        comparison_rows.append({
            "setup_name": setup_name,
            "threshold": shown_threshold,
            "mean_first_experiment_to_reach": float(np.mean(finite_hits)) if finite_hits else np.nan,
            "n_runs_reaching_threshold": len(finite_hits),
            "n_runs_total": len(hit_steps),
        })

    comparison_threshold_summary = pd.DataFrame(comparison_rows)
    for column in ["n_runs_reaching_threshold", "n_runs_total"]:
        comparison_threshold_summary[column] = comparison_threshold_summary[column].astype("Int64")

    print("Single-run threshold result")
    display(single_threshold_summary)
    print("Repeated-run threshold summary")
    display(comparison_threshold_summary)


## Going further: one BO optimizer for multiple datasets

This step is considered optional and an extension over what is expected.

A BO setup can look good on one dataset simply because it matches that dataset well. A **more robust optimizer** is one that performs reasonably well across a **range of datasets and oracles**.

That means the question changes from:

- which BO setup is best for *this* dataset?

to:

- which BO setup is consistently strong across *several* chemistry problems?

A sensible next step would be to keep the same BO setups, run them on several oracle keys, and compare them by the same metrics used above, such as final best value, best-so-far by experiment `n`, or experiments needed to cross a threshold.


In [ ]:
"""
Idea:
1. Choose several ORACLE_KEY values.
2. For each oracle, use its default target.
3. Keep the same named BO setups across all of them.
4. Run the same comparison workflow for each oracle.
5. Collect one summary table per oracle.
6. Compare which setup is most consistently strong overall.

Example sketch:

oracle_keys = ["dataset_a1", "dataset_b4", "dataset_d1"]
all_oracle_summaries = []

for oracle_key in oracle_keys:
    meta = get_oracle_meta(oracle_key)
    bundle = load_oracle(oracle_key)
    target = meta["default_target"]

    results = run_named_setup_comparisons(
        bundle,
        target=target,
        base_config=BASE_WORKBOOK_CONFIG,
        comparison_setups=validated_comparison_setups,
        base_setup_name="main",
    )

    summary = summarize_named_setup_comparisons(
        results,
        compare_after_experiment=COMPARE_AFTER_EXPERIMENT,
    )
    summary.insert(0, "oracle_key", oracle_key)
    summary.insert(1, "oracle_label", meta["label"])
    all_oracle_summaries.append(summary)

overall_summary = pd.concat(all_oracle_summaries, ignore_index=True)
display(overall_summary)

Then you could ask:
- which setup wins most often?
- which setup has the lowest mean regret across datasets?
- which setup is most reliable, even if it is not always the very best?
"""


## Final checklist to evaluate BO

To evaluate your BO setup, ask:

- Did you constrain the search space if necessary? Are all of your suggestions practical and could be executed?
- Did BO improve quickly, or only late in the budget? Why might that have been?
- Did different repeated runs behave similarly or was there large variation? Why?
- Did one BO choice clearly outperform the others, or were several effectively similar?
